In [17]:
!git clone --branch Project_Update2 --single-branch \
  https://github.com/GustavoPCS/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project.git

%cd Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project

!git branch --show-current
!pip -q install transformers accelerate pandas numpy scikit-learn datasets pytest

Cloning into 'Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 119 (delta 38), reused 70 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 864.44 KiB | 6.45 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project
Project_Update2


In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss
)
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit

ROBERTA_DIR = Path("results/roberta_seed42")
FAST_DIR = Path("results/fast_detectgpt")
OUT_DIR = Path("results/threshold_analysis_person_b")
OUT_DIR.mkdir(parents=True, exist_ok=True)

roberta_val = pd.read_csv(ROBERTA_DIR / "predictions_val.csv")
roberta_test = pd.read_csv(ROBERTA_DIR / "predictions_test.csv")

fast_val = pd.read_csv(FAST_DIR / "predictions_val.csv")
fast_test = pd.read_csv(FAST_DIR / "predictions_test.csv")

print("RoBERTa validation/test:", roberta_val.shape, roberta_test.shape)
print("Fast-DetectGPT validation/test:", fast_val.shape, fast_test.shape)

RoBERTa validation/test: (786, 9) (6088, 9)
Fast-DetectGPT validation/test: (786, 9) (6088, 9)


In [19]:
CALIBRATION_COLS = ["method", "train_domain", "train_generator"]

def threshold_max_f1(y, scores):
    candidates = np.unique(scores)
    values = [
        f1_score(y, scores >= threshold, average="macro", zero_division=0)
        for threshold in candidates
    ]
    return float(candidates[np.argmax(values)])

def threshold_low_fpr(y, scores, target_fpr=0.01):
    human_scores = np.asarray(scores)[np.asarray(y) == 0]

    candidates = np.unique(
        np.concatenate([scores, [np.nextafter(np.max(scores), np.inf)]])
    )

    valid = [
        threshold
        for threshold in candidates
        if np.mean(human_scores >= threshold) <= target_fpr
    ]

    return float(min(valid))

def threshold_midpoint_median(y, scores):
    human_median = np.median(np.asarray(scores)[np.asarray(y) == 0])
    machine_median = np.median(np.asarray(scores)[np.asarray(y) == 1])

    return float((human_median + machine_median) / 2)

def classification_summary(y, scores, threshold):
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores, dtype=float)
    predictions = (scores >= threshold).astype(int)

    tp = ((y == 1) & (predictions == 1)).sum()
    fp = ((y == 0) & (predictions == 1)).sum()
    fn = ((y == 1) & (predictions == 0)).sum()
    tn = ((y == 0) & (predictions == 0)).sum()

    return {
        "n": len(y),
        "n_human": int((y == 0).sum()),
        "n_machine": int((y == 1).sum()),
        "threshold": float(threshold),
        "macro_f1": f1_score(y, predictions, average="macro", zero_division=0),
        "accuracy": accuracy_score(y, predictions),
        "auroc": roc_auc_score(y, scores),
        "tpr": tp / (tp + fn) if (tp + fn) else np.nan,
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
    }

def scenario_name(row):
    same_domain = row["domain"] == row["train_domain"]
    same_generator = row["generator"] == row["train_generator"]

    if same_domain and same_generator:
        return "in_domain"
    if not same_domain and same_generator:
        return "domain_shift"
    if same_domain and not same_generator:
        return "generator_shift"
    return "combined_shift"

In [20]:
def analyze_threshold_rules(val_df, test_df, detector_name, score_column="score"):
    threshold_rows = []
    result_rows = []

    for key, val_group in val_df.groupby(CALIBRATION_COLS):
        method, train_domain, train_generator = key

        y_val = val_group["label"].to_numpy()
        scores_val = val_group[score_column].to_numpy()

        rules = {
            "max_f1": threshold_max_f1(y_val, scores_val),
            "low_fpr_1pct": threshold_low_fpr(y_val, scores_val, target_fpr=0.01),
            "midpoint_median": threshold_midpoint_median(y_val, scores_val),
        }

        matching_test = test_df[
            (test_df["method"] == method) &
            (test_df["train_domain"] == train_domain) &
            (test_df["train_generator"] == train_generator)
        ]

        for rule, threshold in rules.items():
            threshold_rows.append({
                "detector": detector_name,
                "method": method,
                "train_domain": train_domain,
                "train_generator": train_generator,
                "rule": rule,
                "threshold": threshold,
            })

            for (domain, generator), test_group in matching_test.groupby(
                ["domain", "generator"]
            ):
                metrics = classification_summary(
                    test_group["label"],
                    test_group[score_column],
                    threshold
                )

                result_rows.append({
                    "detector": detector_name,
                    "method": method,
                    "train_domain": train_domain,
                    "train_generator": train_generator,
                    "domain": domain,
                    "generator": generator,
                    "rule": rule,
                    **metrics
                })

    results = pd.DataFrame(result_rows)
    thresholds = pd.DataFrame(threshold_rows)

    results["scenario"] = results.apply(scenario_name, axis=1)

    baseline = results[
        results["scenario"] == "in_domain"
    ][[
        "detector", "method", "train_domain",
        "train_generator", "rule", "macro_f1"
    ]].rename(columns={"macro_f1": "in_domain_macro_f1"})

    results = results.merge(
        baseline,
        on=[
            "detector", "method", "train_domain",
            "train_generator", "rule"
        ],
        how="left"
    )

    results["gap_macro_f1"] = (
        results["in_domain_macro_f1"] - results["macro_f1"]
    )

    return thresholds, results

In [21]:
roberta_thresholds, roberta_results = analyze_threshold_rules(
    roberta_val,
    roberta_test,
    detector_name="RoBERTa"
)

fast_thresholds, fast_results = analyze_threshold_rules(
    fast_val,
    fast_test,
    detector_name="Fast-DetectGPT"
)

raw_thresholds = pd.concat(
    [roberta_thresholds, fast_thresholds],
    ignore_index=True
)

raw_results = pd.concat(
    [roberta_results, fast_results],
    ignore_index=True
)

display(
    raw_results[
        [
            "detector", "rule", "train_domain", "domain", "generator",
            "threshold", "macro_f1", "auroc", "tpr", "fpr",
            "gap_macro_f1"
        ]
    ].round(3)
)

,detector,rule,train_domain,domain,generator,threshold,macro_f1,auroc,tpr,fpr,gap_macro_f1
0,RoBERTa,max_f1,abstracts,abstracts,llama-chat,0.999,0.978,0.998,0.978,0.022,0.022
1,RoBERTa,max_f1,abstracts,abstracts,mistral-chat,0.999,1.000,1.000,1.000,0.000,0.000
2,RoBERTa,max_f1,abstracts,news,llama-chat,0.999,0.884,0.963,0.851,0.082,0.116
3,RoBERTa,max_f1,abstracts,news,mistral-chat,0.999,0.829,0.927,0.768,0.109,0.171
4,RoBERTa,max_f1,abstracts,reddit,llama-chat,0.999,0.767,0.915,0.576,0.024,0.233
...,...,...,...,...,...,...,...,...,...,...,...
187,Fast-DetectGPT,midpoint_median,reviews,news,mistral-chat,3.273,1.000,1.000,1.000,0.000,0.000
188,Fast-DetectGPT,midpoint_median,reviews,reddit,llama-chat,3.273,0.988,1.000,0.976,0.000,0.012
189,Fast-DetectGPT,midpoint_median,reviews,reddit,mistral-chat,3.273,0.990,1.000,0.980,0.000,0.010
190,Fast-DetectGPT,midpoint_median,reviews,reviews,llama-chat,3.273,1.000,1.000,1.000,0.000,0.000


In [22]:
from scipy.optimize import minimize_scalar
from scipy.special import expit

def safe_logit(probabilities):
    """
    Convert saved probabilities to log-odds without broad clipping.

    This preserves the ordering of all non-identical saved probabilities.
    Exact 0/1 values are replaced only by the nearest representable float.
    """
    p = np.asarray(probabilities, dtype=np.float64).copy()

    lower = np.nextafter(0.0, 1.0)
    upper = np.nextafter(1.0, 0.0)

    p[p <= 0.0] = lower
    p[p >= 1.0] = upper

    return np.log(p) - np.log1p(-p)


def temperature_nll(logits, labels, temperature):
    """
    Numerically stable binary negative log-likelihood.
    """
    scaled_logits = logits / temperature
    labels = np.asarray(labels, dtype=np.float64)

    return np.mean(
        np.logaddexp(0.0, scaled_logits) - labels * scaled_logits
    )


def fit_temperature(scores, labels):
    """
    Learn one positive temperature from validation predictions only.
    """
    logits = safe_logit(scores)

    result = minimize_scalar(
        lambda temperature: temperature_nll(logits, labels, temperature),
        bounds=(0.05, 10.0),
        method="bounded"
    )

    return float(result.x)


def add_temperature_scaled_logits(val_df, test_df):
    val_scaled = val_df.copy()
    test_scaled = test_df.copy()

    # Use scaled LOGITS for analysis, not re-converted probabilities.
    # This prevents loss of ranking from numerical saturation.
    val_scaled["score_temp_logit"] = np.nan
    test_scaled["score_temp_logit"] = np.nan

    temperature_rows = []

    for key, val_group in val_df.groupby(CALIBRATION_COLS):
        method, train_domain, train_generator = key

        temperature = fit_temperature(
            val_group["score"].to_numpy(),
            val_group["label"].to_numpy()
        )

        val_mask = (
            (val_scaled["method"] == method) &
            (val_scaled["train_domain"] == train_domain) &
            (val_scaled["train_generator"] == train_generator)
        )

        test_mask = (
            (test_scaled["method"] == method) &
            (test_scaled["train_domain"] == train_domain) &
            (test_scaled["train_generator"] == train_generator)
        )

        val_logits = safe_logit(
            val_scaled.loc[val_mask, "score"].to_numpy()
        )

        test_logits = safe_logit(
            test_scaled.loc[test_mask, "score"].to_numpy()
        )

        val_scaled.loc[val_mask, "score_temp_logit"] = val_logits / temperature
        test_scaled.loc[test_mask, "score_temp_logit"] = test_logits / temperature

        # These are calibration diagnostics, not the threshold-analysis scores.
        raw_probabilities = val_scaled.loc[val_mask, "score"].to_numpy()
        scaled_probabilities = expit(np.clip(val_logits / temperature, -700, 700))
        labels = val_scaled.loc[val_mask, "label"].to_numpy()

        temperature_rows.append({
            "method": method,
            "train_domain": train_domain,
            "train_generator": train_generator,
            "temperature": temperature,
            "raw_brier": brier_score_loss(labels, raw_probabilities),
            "temperature_scaled_brier": brier_score_loss(
                labels,
                scaled_probabilities
            ),
            "raw_nll": log_loss(labels, raw_probabilities, labels=[0, 1]),
            "temperature_scaled_nll": log_loss(
                labels,
                scaled_probabilities,
                labels=[0, 1]
            ),
        })

    return val_scaled, test_scaled, pd.DataFrame(temperature_rows)


roberta_val_temp, roberta_test_temp, temperature_table = (
    add_temperature_scaled_logits(roberta_val, roberta_test)
)

display(temperature_table.round(4))

,method,train_domain,train_generator,temperature,raw_brier,temperature_scaled_brier,raw_nll,temperature_scaled_nll
0,roberta_base_seed42,abstracts,mistral-chat,1.4345,0.0147,0.0128,0.0480,0.0410
1,roberta_base_seed42,news,mistral-chat,0.0500,0.0000,0.0000,0.0002,0.0000
2,roberta_base_seed42,reddit,mistral-chat,1.5166,0.0105,0.0099,0.0504,0.0403
3,roberta_base_seed42,reviews,mistral-chat,1.3204,0.0053,0.0050,0.0211,0.0186


In [23]:
roberta_temp_thresholds, roberta_temp_results = analyze_threshold_rules(
    roberta_val_temp,
    roberta_test_temp,
    detector_name="RoBERTa_temperature_scaled",
    score_column="score_temp_logit"
)

# AUROC must match raw RoBERTa because temperature scaling preserves ranking.
raw_auroc = roberta_results[
    roberta_results["rule"] == "max_f1"
][[
    "method", "train_domain", "train_generator",
    "domain", "generator", "auroc"
]].rename(columns={"auroc": "raw_auroc"})

temp_auroc = roberta_temp_results[
    roberta_temp_results["rule"] == "max_f1"
][[
    "method", "train_domain", "train_generator",
    "domain", "generator", "auroc"
]].rename(columns={"auroc": "temperature_scaled_auroc"})

auroc_check = raw_auroc.merge(
    temp_auroc,
    on=[
        "method", "train_domain", "train_generator",
        "domain", "generator"
    ]
)

auroc_check["difference"] = (
    auroc_check["temperature_scaled_auroc"] -
    auroc_check["raw_auroc"]
)

display(auroc_check.round(12))

assert np.allclose(
    auroc_check["raw_auroc"],
    auroc_check["temperature_scaled_auroc"],
    equal_nan=True
), "AUROC changed: inspect the score transformation."

,method,train_domain,train_generator,domain,generator,raw_auroc,temperature_scaled_auroc,difference
0,roberta_base_seed42,abstracts,mistral-chat,abstracts,llama-chat,0.997637,0.997637,0.0
1,roberta_base_seed42,abstracts,mistral-chat,abstracts,mistral-chat,1.000000,1.000000,0.0
2,roberta_base_seed42,abstracts,mistral-chat,news,llama-chat,0.962965,0.962965,0.0
3,roberta_base_seed42,abstracts,mistral-chat,news,mistral-chat,0.927116,0.927116,0.0
4,roberta_base_seed42,abstracts,mistral-chat,reddit,llama-chat,0.915294,0.915294,0.0
5,roberta_base_seed42,abstracts,mistral-chat,reddit,mistral-chat,0.975870,0.975870,0.0
6,roberta_base_seed42,abstracts,mistral-chat,reviews,llama-chat,0.981504,0.981504,0.0
7,roberta_base_seed42,abstracts,mistral-chat,reviews,mistral-chat,0.969993,0.969993,0.0
8,roberta_base_seed42,news,mistral-chat,abstracts,llama-chat,0.968809,0.968809,0.0
9,roberta_base_seed42,news,mistral-chat,abstracts,mistral-chat,0.952393,0.952393,0.0


In [24]:
all_thresholds = pd.concat(
    [raw_thresholds, roberta_temp_thresholds],
    ignore_index=True
)

all_results = pd.concat(
    [raw_results, roberta_temp_results],
    ignore_index=True
)

summary = (
    all_results
    .groupby(["detector", "rule", "scenario"], as_index=False)
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        mean_auroc=("auroc", "mean"),
        mean_tpr=("tpr", "mean"),
        mean_fpr=("fpr", "mean"),
        mean_gap_macro_f1=("gap_macro_f1", "mean"),
        n_conditions=("macro_f1", "size"),
    )
)

display(summary.round(3))

print(
    summary.pivot_table(
        index=["detector", "scenario"],
        columns="rule",
        values=["mean_macro_f1", "mean_tpr", "mean_fpr", "mean_gap_macro_f1"]
    ).round(3).to_latex()
)

,detector,rule,scenario,mean_macro_f1,mean_auroc,mean_tpr,mean_fpr,mean_gap_macro_f1,n_conditions
0,Fast-DetectGPT,low_fpr_1pct,combined_shift,0.992,1.000,0.984,0.000,0.006,12
1,Fast-DetectGPT,low_fpr_1pct,domain_shift,0.997,1.000,0.994,0.000,0.000,12
2,Fast-DetectGPT,low_fpr_1pct,generator_shift,0.993,1.000,0.985,0.000,0.005,4
3,Fast-DetectGPT,low_fpr_1pct,in_domain,0.997,1.000,0.995,0.000,0.000,4
4,Fast-DetectGPT,max_f1,combined_shift,0.985,1.000,0.969,0.000,0.013,12
5,Fast-DetectGPT,max_f1,domain_shift,0.994,1.000,0.988,0.000,0.003,12
6,Fast-DetectGPT,max_f1,generator_shift,0.993,1.000,0.985,0.000,0.005,4
7,Fast-DetectGPT,max_f1,in_domain,0.997,1.000,0.995,0.000,0.000,4
8,Fast-DetectGPT,midpoint_median,combined_shift,0.992,1.000,0.985,0.002,0.006,12
9,Fast-DetectGPT,midpoint_median,domain_shift,0.997,1.000,0.995,0.001,0.000,12


\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 &  & \multicolumn{3}{r}{mean_fpr} & \multicolumn{3}{r}{mean_gap_macro_f1} & \multicolumn{3}{r}{mean_macro_f1} & \multicolumn{3}{r}{mean_tpr} \\
 & rule & low_fpr_1pct & max_f1 & midpoint_median & low_fpr_1pct & max_f1 & midpoint_median & low_fpr_1pct & max_f1 & midpoint_median & low_fpr_1pct & max_f1 & midpoint_median \\
detector & scenario &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{Fast-DetectGPT} & combined_shift & 0.000000 & 0.000000 & 0.002000 & 0.006000 & 0.013000 & 0.006000 & 0.992000 & 0.985000 & 0.992000 & 0.984000 & 0.969000 & 0.985000 \\
 & domain_shift & 0.000000 & 0.000000 & 0.001000 & 0.000000 & 0.003000 & 0.000000 & 0.997000 & 0.994000 & 0.997000 & 0.994000 & 0.988000 & 0.995000 \\
 & generator_shift & 0.000000 & 0.000000 & 0.000000 & 0.005000 & 0.005000 & -0.001000 & 0.993000 & 0.993000 & 0.999000 & 0.985000 & 0.985000 & 0.997000 \\
 & in_domain & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.000000 & 0.00

In [25]:
all_thresholds.to_csv(
    OUT_DIR / "thresholds_by_rule.csv",
    index=False
)

all_results.to_csv(
    OUT_DIR / "threshold_rule_results.csv",
    index=False
)

summary.to_csv(
    OUT_DIR / "threshold_rule_summary.csv",
    index=False
)

temperature_table.to_csv(
    OUT_DIR / "roberta_temperature_scaling.csv",
    index=False
)

print("Saved files:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path.name)

Saved files:
- roberta_temperature_scaling.csv
- threshold_rule_results.csv
- threshold_rule_summary.csv
- thresholds_by_rule.csv


In [26]:
from google.colab import files
import shutil

# Creates one ZIP containing all threshold-analysis CSV files.
zip_path = shutil.make_archive(
    "threshold_analysis_person_b",
    "zip",
    root_dir=OUT_DIR
)

print(f"Created: {zip_path}")
files.download(zip_path)

Created: /content/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/Detecting-AI-Generated-Text-Across-Domains-and-Generators---NLP-Project/threshold_analysis_person_b.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>